# LASSER on Kaggle

GEARS with an optional GSR-style learned co-expression graph. Needs **Internet on** (GEARS downloads Norman and the GO graph from Harvard Dataverse) and a **GPU (T4)**.

Cells: install, get the code, run the tests (before any real training), baseline run, graph-learning run, results.

In [1]:
# 1. Dependencies. GEARS is vendored inside lasser/ (lasser/gears), so no cell-gears install.
#    Kaggle already has torch, numpy, pandas, scipy, scikit-learn, networkx, tqdm, requests.
!pip install -q torch_geometric scanpy pytest

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 57.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.2/190.2 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 376.9/376.9 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.5/230.5 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 1.9 MB/s eta 0:00:00


In [2]:
# 2. Get the code: clone, then move the self-contained package next to the notebook.
REPO_URL = "https://github.com/LegendaryDarkKnight/LASSER"   # <- set this
REPO_DIR = "/kaggle/working/lasser-repo"
!rm -rf {REPO_DIR} /kaggle/working/lasser
!git clone -q {REPO_URL} {REPO_DIR}
!mv {REPO_DIR}/lasser /kaggle/working/lasser
!git -C {REPO_DIR} log -1 --oneline

c7be65c (HEAD -> main, origin/main, origin/HEAD) minor fix


In [3]:
# 3. Tests 1-8 (tiny subsets; first run also downloads Norman, ~10-20 min in total).
%env LASSER_DATA_DIR=/kaggle/working/data
%env LASSER_TEST_DIR=/kaggle/working/lasser_tests
!cd /kaggle/working && python -m pytest {REPO_DIR}/tests -m "not smoke" -p no:cacheprovider -rA -s --tb=short 2>&1 | tee /kaggle/working/test_output.txt | grep -vE "it/s\]|s/it\]"

env: LASSER_DATA_DIR=/kaggle/working/data
env: LASSER_TEST_DIR=/kaggle/working/lasser_tests
============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0
rootdir: /kaggle/working
plugins: fast-array-utils-1.5.1, zarr-3.4.0, typeguard-4.6.0, anyio-4.14.2, langsmith-0.12.1
collected 15 items

lasser-repo/tests/test_00_packaging.py EEE
lasser-repo/tests/test_01_parity.py Found local copy...
Found local copy...
Found local copy...
These perturbations are not in the GO graph and their perturbation can thus not be predicted
['RHOXF2BB+ctrl' 'LYL1+IER5L' 'ctrl+IER5L' 'KIAA1804+ctrl' 'IER5L+ctrl'
 'RHOXF2BB+ZBTB25' 'RHOXF2BB+SET']
Local copy of pyg dataset is detected. Loading...
Done!
Local copy of split is detected. Loading...
Simulation split test composition:
combo_seen0:9
combo_seen1:43
combo_seen2:19
unseen_single:36
Done!
Creating dataloaders....
Done!
Found local copy...
Local copy of split is detected. Loa

In [4]:
# 3b. Optional test 9: full Norman, 1 epoch, flag off and on, checks the T4 memory budget.
%env LASSER_SMOKE=1
!cd /kaggle/working && python -m pytest {REPO_DIR}/tests/test_09_smoke.py -p no:cacheprovider -rA -s --tb=short 2>&1 | tee /kaggle/working/smoke_output.txt | grep -vE "it/s\]|s/it\]"
%env LASSER_SMOKE=0

env: LASSER_SMOKE=1
============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0
rootdir: /kaggle/working
plugins: fast-array-utils-1.5.1, zarr-3.4.0, typeguard-4.6.0, anyio-4.14.2, langsmith-0.12.1
collected 2 items

lasser-repo/tests/test_09_smoke.py Found local copy...
Found local copy...
Found local copy...
These perturbations are not in the GO graph and their perturbation can thus not be predicted
['RHOXF2BB+ctrl' 'LYL1+IER5L' 'ctrl+IER5L' 'KIAA1804+ctrl' 'IER5L+ctrl'
 'RHOXF2BB+ZBTB25' 'RHOXF2BB+SET']
Local copy of pyg dataset is detected. Loading...
Done!
2026-10-06 09:23:03,956 INFO lasser.tracking: run 402d2197 -> /kaggle/working/lasser_tests/runs/norman_seed1_base_20261006-092303_smoke
Local copy of split is detected. Loading...
Simulation split test composition:
combo_seen0:9
combo_seen1:43
combo_seen2:19
unseen_single:36
Done!
Creating dataloaders....
Done!
here1
2026-10-06 09:23:04,033 INFO la

In [5]:
# 4. Load the data once and reuse it for both runs.
from lasser import GSRConfig, LasserConfig, load_pert_data, load_run, run_experiment

common = dict(dataset="norman", seed=1, epochs=20,
              data_dir="/kaggle/working/data", out_dir="/kaggle/working/runs",
              cache_dir="/kaggle/working/cache", repo_dir=REPO_DIR)
pert_data = load_pert_data(LasserConfig(**common))

Found local copy...
Found local copy...
Found local copy...
These perturbations are not in the GO graph and their perturbation can thus not be predicted
['RHOXF2BB+ctrl' 'LYL1+IER5L' 'ctrl+IER5L' 'KIAA1804+ctrl' 'IER5L+ctrl'
 'RHOXF2BB+ZBTB25' 'RHOXF2BB+SET']
Local copy of pyg dataset is detected. Loading...
Done!


In [6]:
# 5. Baseline: graph_learning=False is plain GEARS (plus logging and saving).
base = run_experiment(LasserConfig(**common, graph_learning=False), pert_data)
base["run_dir"]

2026-10-06 09:36:49,611 INFO lasser.tracking: run dedd34cd -> /kaggle/working/runs/norman_seed1_base_20261006-093649


Local copy of split is detected. Loading...
Simulation split test composition:
combo_seen0:9
combo_seen1:43
combo_seen2:19
unseen_single:36
Done!
Creating dataloaders....
Done!


here1
2026-10-06 09:36:49,697 INFO lasser.tracking: split hash f93e7d59ee239960 (copied ['norman_simulation_1_0.75_subgroup.pkl', 'norman_simulation_1_0.75.pkl'])
2026-10-06 09:36:49,700 INFO lasser.utils: [gears_init] start


Found local copy...


2026-10-06 09:37:10,907 INFO lasser.utils: [gears_init] done in 21.2s, peak CUDA memory 0.03 GB
2026-10-06 09:37:10,912 INFO lasser.run: co-expression graph used (gears_static): {'num_nodes': 5045, 'num_edges': 6364, 'num_self_loops': 4699, 'degree_min': 0, 'degree_median': 0.0, 'degree_mean': 0.33002973240832506, 'degree_max': 20, 'degree_gini': 0.9324442777762897, 'isolated_genes': 4419}
2026-10-06 09:37:10,912 INFO lasser.utils: [gears_train] start


Start Training...
Epoch 1 Step 1 Train Loss: 0.5244
Epoch 1 Step 51 Train Loss: 0.6157
Epoch 1 Step 101 Train Loss: 0.5193
Epoch 1 Step 151 Train Loss: 0.5159
Epoch 1 Step 201 Train Loss: 0.4628
Epoch 1 Step 251 Train Loss: 0.5926
Epoch 1 Step 301 Train Loss: 0.4182
Epoch 1 Step 351 Train Loss: 0.5559
Epoch 1 Step 401 Train Loss: 0.5089
Epoch 1 Step 451 Train Loss: 0.5312
Epoch 1 Step 501 Train Loss: 0.4393
Epoch 1 Step 551 Train Loss: 0.6198
Epoch 1 Step 601 Train Loss: 0.4603
Epoch 1 Step 651 Train Loss: 0.5034
Epoch 1 Step 701 Train Loss: 0.4582
Epoch 1 Step 751 Train Loss: 0.4548
Epoch 1 Step 801 Train Loss: 0.4768
Epoch 1 Step 851 Train Loss: 0.4305
Epoch 1 Step 901 Train Loss: 0.5522
Epoch 1 Step 951 Train Loss: 0.4326
Epoch 1 Step 1001 Train Loss: 0.4202
Epoch 1 Step 1051 Train Loss: 0.4902
Epoch 1 Step 1101 Train Loss: 0.5046
Epoch 1 Step 1151 Train Loss: 0.4284
Epoch 1 Step 1201 Train Loss: 0.4323
Epoch 1 Step 1251 Train Loss: 0.4354
Epoch 1 Step 1301 Train Loss: 0.4792
Epoch 

2026-10-06 11:03:10,981 INFO lasser.utils: [gears_train] done in 5160.1s, peak CUDA memory 0.87 GB
2026-10-06 11:03:10,982 ERROR lasser.run: run dedd34cd failed
Traceback (most recent call last):
  File "/kaggle/working/lasser/run.py", line 164, in run_experiment
    return _train_evaluate_save(ctx, tracker)
  File "/kaggle/working/lasser/run.py", line 199, in _train_evaluate_save
    parsed = parse_gears_output(lines)
  File "/kaggle/working/lasser/tracking.py", line 266, in parse_gears_output
    "val_mse": float(m.group(3))})
               ~~~~~^^^^^^^^^^^^
ValueError: could not convert string to float: '0.0099.'


ValueError: could not convert string to float: '0.0099.'

In [ ]:
# 6. Graph learning: GSR-refined co-expression graph (defaults from the plan, section 11).
gl = run_experiment(LasserConfig(**common, graph_learning=True, gsr=GSRConfig()), pert_data)
gl["run_dir"], gl["checks"]["h2"], gl["checks"]["refine"]

In [ ]:
# 7. Results: the global run table, and any past run loaded back.
import pandas as pd
pd.set_option("display.max_columns", 200)
runs = pd.read_csv("/kaggle/working/runs/runs.csv")
cols = ["run_id", "seed", "graph_learning", "gsr_init_gene_emb", "graph_num_edges", "graph_isolated_genes",
        "test_pearson_delta", "test_mse_top20_de", "test_nmse_top20_de",
        "test_unseen_single_pearson_delta", "test_combo_seen0_pearson_delta",
        "test_combo_seen1_pearson_delta", "test_combo_seen2_pearson_delta", "run_dir"]
display(runs[[c for c in cols if c in runs.columns]])

r = load_run(gl["run_dir"])
display(r["epoch_log"])
r["test_metrics"]["paper"]["subgroups"]